### RunnableBranch
- 입력에 따라 동적으로 로직을 라우팅할 수 있는 강력한 도구
- 입력 데이터의 특성에 기반하여 다양한 처리 경로를 유연하게 정의

- 복잡한 의사 결정 트리를 간단하고 직관적인 방식으로 구현
- 런타임에 동적으로 분기 조건을 평가하고 적절한 처리 루틴을 선택하여 시스템의 적응력과 확장성을 높임
- 입력 데이터의 다양성과 변동성이 큰 애플리케이션 개발에 매우 유용

### 입력에 따른 동적 로직 라우팅

- 라우팅을 통해 이전 단계의 출력이 다음 단계를 정의하는 비결정적 체인생성 가능

In [1]:
from dotenv import load_dotenv

load_dotenv()

from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
     """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)


chain = (prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser())

In [3]:
chain.invoke({"question" : "2+2 는 무엇인가요 ?"})

'수학'

In [4]:
chain.invoke({"question" : "작용 반작용의 법칙은 무엇인가요?"})

'과학'

In [5]:
chain.invoke({"question" : "Google은 어떤 회사인가요?"})

'기타'

In [6]:
# 3개의 하위 체인 생성

math_chain = (PromptTemplate.from_template(
    """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
) | ChatOpenAI(model="gpt-4o-mini") 
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

In [7]:
# 사용자 정의 함수 사용하기

def route(info):
    # 주제에 "수학"이 포함되어 있는 경우
    if "수학" in info["topic"].lower():
        return math_chain

    elif "과학" in info["topic"].lower():
        return science_chain

    else:
        return general_chain

In [8]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

full_chain = (
    {"topic" : chain, "question" : itemgetter("question")} 
    | RunnableLambda(route) | StrOutputParser()
)

In [9]:
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 함수의 변화를 연구하는 중요한 도구입니다. 미적분은 두 가지 주요 개념인 미분과 적분으로 나뉩니다.\n\n미분은 함수의 변화율을 측정하는 방법입니다. 즉, 어떤 함수가 주어졌을 때, 그 함수의 기울기나 순간적인 변화 속도를 알고 싶을 때 사용합니다. 예를 들어, 자동차의 속도가 시간에 따라 어떻게 변하는지를 알고 싶다면 미분을 통해 순간 속도를 계산할 수 있습니다.\n\n적분은 미분의 역 과정으로, 함수의 면적이나 전체적인 양을 구하는 데 사용됩니다. 예를 들어, 일정 시간 동안 자동차가 이동한 총 거리를 알고 싶다면 그 속도를 적분하여 구할 수 있습니다.\n\n이처럼 미적분은 자연과학, 공학, 경제학 등 다양한 분야에 널리 활용되며, 수학적 모델링과 문제 해결에 필수적인 도구로 자리 잡고 있습니다.'

In [10]:
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력은 두 물체 사이의 질량에 비례하고, 이들 사이의 거리의 제곱에 반비례하는 힘이라고 하셨습니다. 이는 모든 물체가 서로 중력으로 끌어당기며, 따라서 지구와 같은 큰 천체는 그 주위의 작은 물체를 끌어당기는 능력을 가지고 있습니다. 뉴턴의 만유인력 법칙에 의하면, 이 힘은 물체의 질량이 클수록, 그리고 그 거리의 제곱이 작을수록 더욱 강해진다고 하셨습니다. 이러한 중력의 작용 때문에 사물은 지구로 떨어지고, 행성들은 태양 주위를 공전하게 되는 것입니다.'

In [11]:
full_chain.invoke(
    {"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"}
    )

'RAG(Retrieval Augmented Generation)는 정보 검색과 생성 모델을 결합한 접근 방식으로, 주어진 질문에 대해 관련 정보를 검색한 후 이를 바탕으로 텍스트를 생성하는 방법입니다. 이를 통해 더 정확하고 정보 풍부한 응답을 제공할 수 있습니다.'

### RunnableBranch
- 입력값에 따라 실행할 조건과 Runnable을 정의할 수 있는 특별한 유형의 Runnable
- 위에서 설명한 사용자 정의 함수로 구현할 수 없는 기능을 제공하지는 않으므로, 사용자 정의 함수를 사용하는 것이 좋음

In [12]:
from operator import itemgetter
from langchain_core.runnables import RunnableBranch
branch = RunnableBranch(
    # 주제에 수학이 포함되어 있는지 확인
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    # 주제에 과학이 포함되어 있는지 확인
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    general_chain
)

# 주제와 질문을 입력받아 branch를 실행하는 전체 체인을 정의
full_chain = (
    {"topic": chain, "question": itemgetter("question")} | branch | StrOutputParser()
)

In [13]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분과로, 두 가지 주요 개념인 미분과 적분을 포함하고 있습니다. 미분은 함수의 변화율을 다루며, 즉 주어진 함수가 특정 지점에서 얼마나 빠르게 변화하는지, 또는 기울기를 찾는 과정입니다. 적분은 함수의 구간에 걸쳐 면적을 계산하는 것으로, 일반적으로 어떤 양이 어떻게 누적되는지를 나타냅니다. \n\n미적분은 자연 현상을 모델링하거나 최적화 문제를 해결하는 데 필수적인 도구로 사용되며, 물리학, 공학, 경제학 등 다양한 분야에서 광범위하게 응용됩니다. 기본적으로 미적분은 무한소를 다루는 수학적 방법론으로, 연속적인 변화를 수학적으로 정량화하는 데 큰 역할을 하며, 이는 경계가 없는 풍부한 수학적 탐구로 이어집니다.'

In [14]:

# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "중력 가속도는 어떻게 계산하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력 가속도는 지구와 같은 천체가 물체를 끌어당기는 힘을 측정하는 것으로, 일반적으로 심력의 가속도는 약 9.81m/s²로 정의됩니다. 이를 계산하기 위해서는 만유인력 법칙을 사용할 수 있습니다. 만유인력 법칙에 따르면 두 물체 사이의 중력은 물체의 질량과 거리의 제곱에 반비례합니다. 중력 가속도(g)는 다음과 같이 계산됩니다:\n\n\\[ g = \\frac{GM}{r^2} \\]\n\n여기서 G는 만유인력 상수, M은 지구의 질량, r은 지구의 반지름입니다. 이 공식을 사용하면 특정 위치에서의 중력 가속도를 정확하게 구할 수 있습니다.'

In [15]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(Retrieval Augmented Generation)는 정보 검색 기능과 생성 모델을 결합한 자연어 처리 기법으로, 사용자의 질문에 대한 정확한 답변을 생성하기 위해 외부 데이터베이스에서 관련 정보를 검색한 후 이를 바탕으로 텍스트를 생성합니다.'